# Multi-Agent System with MCP Integration and Execution Tracing

This notebook is a self-contained technical demonstration combining the Model Context
Protocol (MCP), a supervisor–worker multi-agent architecture, and a custom execution
tracing layer. It is designed to be read and run sequentially, with each stage building
on the one before it.

## Objective

The notebook implements and demonstrates the following capabilities in a single, runnable
environment:

- A custom MCP server exposing one application resource and one tool
- A client that connects to the MCP server and invokes its resource and tool
- A supervisor agent that routes incoming tasks to two or more specialized worker agents
- A tracing layer that records every tool call, resource read, and routing decision made
  across the agent graph

Each capability is implemented, executed, and verified inline, with its trace output
visible directly in the notebook.

## Requirements

- Python 3.10+
- Internet access, for package installation and MCP server communication
- `ANTHROPIC_API_KEY` (optional) — enables the `GeneralWorker` fallback agent to call a
  live Claude model instead of returning a canned response; all other functionality
  works without it

Cells are meant to be executed sequentially, top to bottom.

## 1. Environment Setup

Install the packages required for the MCP server, MCP client, agent implementation, and
data handling. The install helper falls back to `--break-system-packages` for systems
that enforce PEP 668 (externally managed environments).

In [1]:
import subprocess, sys

def pip_install(*packages):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    except subprocess.CalledProcessError:
        # Fallback for PEP 668 "externally managed environment" systems
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--break-system-packages", *packages
        ])

pip_install(
    "mcp",
    "anthropic",
    "nest_asyncio",
    "pandas",
    "pywin32"
)

## 2. MCP Server Implementation

An MCP server runs as an independent process communicating over stdio, so it is written
to a standalone file, `mcp_server.py`, using `FastMCP`. The server exposes exactly one
resource and one tool:

| Type | Name | Description |
|---|---|---|
| Resource | `app://status` | Returns application metadata and health status as JSON |
| Tool | `search_knowledge_base(query)` | Full-text search over a small in-memory knowledge base |

In [2]:
%%writefile mcp_server.py
"""
Custom MCP server.

Exposes:
  - 1 resource : app://status                 -> app metadata / health info
  - 1 tool     : search_knowledge_base(query)  -> search a small in-memory KB
"""

import json
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoApp")

KNOWLEDGE_BASE = [
    {"id": "kb1", "topic": "refund policy",   "text": "Refunds are available within 30 days of purchase with a valid receipt."},
    {"id": "kb2", "topic": "shipping",        "text": "Standard shipping takes 5-7 business days. Express shipping takes 1-2 business days."},
    {"id": "kb3", "topic": "account password","text": "To reset your password, go to Settings > Security > Reset Password."},
    {"id": "kb4", "topic": "support hours",   "text": "Customer support is available Monday-Friday, 9am-6pm EST."},
    {"id": "kb5", "topic": "warranty",        "text": "All products include a 1-year limited warranty covering manufacturing defects."},
]


@mcp.resource("app://status")
def app_status() -> str:
    """App resource: current application status and metadata, as JSON."""
    return json.dumps(
        {
            "app_name": "DemoApp",
            "version": "1.0.0",
            "status": "healthy",
            "kb_entries": len(KNOWLEDGE_BASE),
        },
        indent=2,
    )


@mcp.tool()
def search_knowledge_base(query: str) -> str:
    """Search the internal knowledge base for entries relevant to the query string."""
    words = query.lower().split()
    matches = [
        f"[{item['topic']}] {item['text']}"
        for item in KNOWLEDGE_BASE
        if any(w in item["topic"] or w in item["text"].lower() for w in words)
    ]
    return "\n".join(matches) if matches else "No matching knowledge base entries found."


if __name__ == "__main__":
    mcp.run()


Writing mcp_server.py


## 3. MCP Client Connection

A custom Python client spawns `mcp_server.py` as a subprocess and communicates with it
over stdio using the official `mcp` SDK. A single `ClientSession` is opened via an
`AsyncExitStack` and kept alive for the remainder of the notebook, to be closed
explicitly in the Cleanup section.

An alternative client setup using Claude Code is described in Section 8; the same server
process works unmodified with either client.

In [3]:
import asyncio
import sys
from contextlib import AsyncExitStack

import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

exit_stack = AsyncExitStack()

async def connect_to_server():
    server_params = StdioServerParameters(
        command=sys.executable,       # run the server with the same Python interpreter
        args=["mcp_server.py"],
    )

    errlog = exit_stack.enter_context(open("mcp_server_stderr.log", "w", buffering=1))

    read, write = await exit_stack.enter_async_context(
        stdio_client(server_params, errlog=errlog)
    )
    session = await exit_stack.enter_async_context(ClientSession(read, write))
    await session.initialize()
    return session

session = await connect_to_server()
print("Connected to MCP server 'DemoApp'.")

tools_response = await session.list_tools()
resources_response = await session.list_resources()

print("\nAvailable tools:")
for t in tools_response.tools:
    print(f"  - {t.name}: {t.description}")

print("\nAvailable resources:")
for r in resources_response.resources:
    print(f"  - {r.uri}  ({r.name})")

Connected to MCP server 'DemoApp'.

Available tools:
  - search_knowledge_base: Search the internal knowledge base for entries relevant to the query string.

Available resources:
  - app://status  (app_status)


## 4. Tracing Layer

`TRACE_LOG` is a single list that every component of the system writes to. Four
categories of events are captured:

| Event Type | Logged By |
|---|---|
| MCP tool call | `call_mcp_tool_traced` |
| MCP resource read | `read_mcp_resource_traced` |
| Local (non-MCP) tool call | `local_tool_traced` |
| Supervisor routing decision | `SupervisorAgent.route` |

Each entry records the calling agent, event type, name, input, output, and duration —
sufficient to reconstruct the complete call graph across the agent system after the
fact.

In [4]:
import time
import json
import datetime
import re
import os

TRACE_LOG = []

def log_event(agent, kind, name, input_data, output_data, duration_ms, meta=None):
    entry = {
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="milliseconds"),
        "agent": agent,
        "kind": kind,          # "mcp_tool_call" | "mcp_resource_read" | "local_tool_call" | "routing_decision"
        "name": name,
        "input": input_data,
        "output": output_data,
        "duration_ms": round(duration_ms, 2),
    }
    if meta:
        entry["meta"] = meta
    TRACE_LOG.append(entry)
    print(f"[TRACE] agent={agent:<16} kind={kind:<18} name={name:<24} duration={entry['duration_ms']}ms")
    return entry


async def call_mcp_tool_traced(agent, tool_name, arguments):
    start = time.time()
    result = await session.call_tool(tool_name, arguments)
    duration = (time.time() - start) * 1000
    output_text = result.content[0].text if result.content else str(result)
    log_event(agent, "mcp_tool_call", tool_name, arguments, output_text, duration)
    return output_text


async def read_mcp_resource_traced(agent, uri):
    start = time.time()
    result = await session.read_resource(uri)
    duration = (time.time() - start) * 1000
    text = result.contents[0].text if result.contents else str(result)
    log_event(agent, "mcp_resource_read", uri, None, text, duration)
    return text


def local_tool_traced(agent, tool_name, func, *args, **kwargs):
    start = time.time()
    output = func(*args, **kwargs)
    duration = (time.time() - start) * 1000
    log_event(agent, "local_tool_call", tool_name, {"args": args, "kwargs": kwargs}, output, duration)
    return output


### 4.1 Sanity Check

Before introducing the agents, the MCP connection is verified directly: the
`app://status` resource is read and the `search_knowledge_base` tool is called, both
through the traced wrapper functions defined above.

In [5]:
status_text = await read_mcp_resource_traced("manual_test", "app://status")
print("App status resource:\n", status_text)

kb_result = await call_mcp_tool_traced("manual_test", "search_knowledge_base", {"query": "shipping"})
print("\nKnowledge base tool result:\n", kb_result)


[TRACE] agent=manual_test      kind=mcp_resource_read  name=app://status             duration=5.13ms
App status resource:
 {
  "app_name": "DemoApp",
  "version": "1.0.0",
  "status": "healthy",
  "kb_entries": 5
}
[TRACE] agent=manual_test      kind=mcp_tool_call      name=search_knowledge_base    duration=5.81ms

Knowledge base tool result:
 [shipping] Standard shipping takes 5-7 business days. Express shipping takes 1-2 business days.


## 5. Supervisor and Worker Agents

Three specialized worker agents are defined, each responsible for a distinct task
category:

| Worker | Handles | Tool Invoked |
|---|---|---|
| `KnowledgeWorker` | Policy, support, and account questions | MCP tool `search_knowledge_base` |
| `MathWorker` | Arithmetic expressions | Local `calculator` tool |
| `GeneralWorker` | Fallback for all other queries | Local `llm_general_answer` tool (uses Claude if `ANTHROPIC_API_KEY` is set, otherwise a canned reply) |

In [6]:
class WorkerAgent:
    name = "base_worker"

    def can_handle(self, task: str) -> bool:
        raise NotImplementedError

    async def run(self, task: str) -> str:
        raise NotImplementedError


class KnowledgeWorker(WorkerAgent):
    name = "knowledge_worker"
    keywords = ["refund", "policy", "shipping", "account", "password",
                "support", "warranty", "docs", "faq", "help"]

    def can_handle(self, task: str) -> bool:
        t = task.lower()
        return any(k in t for k in self.keywords)

    async def run(self, task: str) -> str:
        answer = await call_mcp_tool_traced(self.name, "search_knowledge_base", {"query": task})
        return f"[KnowledgeWorker] {answer}"


class MathWorker(WorkerAgent):
    name = "math_worker"

    def can_handle(self, task: str) -> bool:
        has_digit = bool(re.search(r"\d", task))
        has_op = any(op in task for op in ["+", "-", "*", "/", "plus", "minus", "times", "divided"])
        return has_digit and has_op

    def _calculate(self, expression: str):
        allowed = set("0123456789+-*/(). ")
        cleaned = "".join(ch for ch in expression if ch in allowed)
        return eval(cleaned, {"__builtins__": {}}, {})

    async def run(self, task: str) -> str:
        result = local_tool_traced(self.name, "calculator", self._calculate, task)
        return f"[MathWorker] {task.strip()} -> {result}"


class GeneralWorker(WorkerAgent):
    name = "general_worker"

    def can_handle(self, task: str) -> bool:
        return True  # catch-all fallback

    def _call_llm(self, task: str) -> str:
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            return ("(No ANTHROPIC_API_KEY set - returning canned response) "
                    "I can't look that up right now, but I can help with knowledge-base "
                    "or math questions.")
        import anthropic
        client = anthropic.Anthropic(api_key=api_key)
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=200,
            messages=[{"role": "user", "content": task}],
        )
        return resp.content[0].text

    async def run(self, task: str) -> str:
        answer = local_tool_traced(self.name, "llm_general_answer", self._call_llm, task)
        return f"[GeneralWorker] {answer}"


### 5.1 Supervisor Implementation

The `SupervisorAgent` selects the first worker whose `can_handle()` method returns
`True` and delegates the task to it. Each routing decision is logged to `TRACE_LOG`
before the task is handed off, keeping the routing history fully auditable alongside the
tool calls.

In [7]:
class SupervisorAgent:
    name = "supervisor"

    def __init__(self, workers):
        self.workers = workers

    def route(self, task: str):
        start = time.time()
        chosen = next((w for w in self.workers if w.can_handle(task)), None)
        duration = (time.time() - start) * 1000
        log_event(self.name, "routing_decision", "route_task", task,
                   chosen.name if chosen else None, duration)
        return chosen

    async def handle(self, task: str) -> str:
        worker = self.route(task)
        if worker is None:
            return "[Supervisor] No worker could handle this task."
        return await worker.run(task)


## 6. Demonstration

The supervisor is instantiated with all three workers and run against four sample tasks,
each exercising a different routing path: knowledge-base lookup, arithmetic, and the
general-purpose fallback.

In [8]:
supervisor = SupervisorAgent([KnowledgeWorker(), MathWorker(), GeneralWorker()])

demo_tasks = [
    "What is your refund policy?",
    "What is 42 * 8 + 10?",
    "How many business days does express shipping take?",
    "Tell me a fun fact about octopuses",
]

for task in demo_tasks:
    print("=" * 70)
    print("TASK:", task)
    result = await supervisor.handle(task)
    print("RESULT:", result)


TASK: What is your refund policy?
[TRACE] agent=supervisor       kind=routing_decision   name=route_task               duration=0.01ms
[TRACE] agent=knowledge_worker kind=mcp_tool_call      name=search_knowledge_base    duration=6.16ms
RESULT: [KnowledgeWorker] [refund policy] Refunds are available within 30 days of purchase with a valid receipt.
[account password] To reset your password, go to Settings > Security > Reset Password.
[support hours] Customer support is available Monday-Friday, 9am-6pm EST.
TASK: What is 42 * 8 + 10?
[TRACE] agent=supervisor       kind=routing_decision   name=route_task               duration=0.08ms
[TRACE] agent=math_worker      kind=local_tool_call    name=calculator               duration=0.04ms
RESULT: [MathWorker] What is 42 * 8 + 10? -> 346
TASK: How many business days does express shipping take?
[TRACE] agent=supervisor       kind=routing_decision   name=route_task               duration=0.0ms
[TRACE] agent=knowledge_worker kind=mcp_tool_call      

## 7. Trace Log Analysis

`TRACE_LOG` is loaded into a pandas DataFrame for inspection, giving a consolidated view
of every MCP call, local tool call, and routing decision made during the demonstration
run.

In [9]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
pd.set_option("display.max_colwidth", 60)
trace_df


,timestamp,agent,kind,name,input,output,duration_ms
0,2026-07-22T05:36:54.606+00:00,manual_test,mcp_resource_read,app://status,None,"{\n ""app_name"": ""DemoApp"",\n ""version"": ""1.0.0"",\n ""s...",5.13
1,2026-07-22T05:36:54.612+00:00,manual_test,mcp_tool_call,search_knowledge_base,{'query': 'shipping'},[shipping] Standard shipping takes 5-7 business days. Ex...,5.81
2,2026-07-22T05:36:54.652+00:00,supervisor,routing_decision,route_task,What is your refund policy?,knowledge_worker,0.01
3,2026-07-22T05:36:54.658+00:00,knowledge_worker,mcp_tool_call,search_knowledge_base,{'query': 'What is your refund policy?'},[refund policy] Refunds are available within 30 days of ...,6.16
4,2026-07-22T05:36:54.658+00:00,supervisor,routing_decision,route_task,What is 42 * 8 + 10?,math_worker,0.08
5,2026-07-22T05:36:54.659+00:00,math_worker,local_tool_call,calculator,"{'args': ('What is 42 * 8 + 10?',), 'kwargs': {}}",346,0.04
6,2026-07-22T05:36:54.659+00:00,supervisor,routing_decision,route_task,How many business days does express shipping take?,knowledge_worker,0.00
7,2026-07-22T05:36:54.665+00:00,knowledge_worker,mcp_tool_call,search_knowledge_base,{'query': 'How many business days does express shipping ...,[refund policy] Refunds are available within 30 days of ...,6.34
8,2026-07-22T05:36:54.665+00:00,supervisor,routing_decision,route_task,Tell me a fun fact about octopuses,general_worker,0.02
9,2026-07-22T05:36:54.665+00:00,general_worker,local_tool_call,llm_general_answer,"{'args': ('Tell me a fun fact about octopuses',), 'kwarg...",(No ANTHROPIC_API_KEY set - returning canned response) I...,0.01


### 7.1 Exporting the Trace Log

The complete trace is also persisted to `trace_log.json` for external review or
downstream analysis.

In [10]:
with open("trace_log.json", "w") as f:
    json.dump(TRACE_LOG, f, indent=2)

print(f"Wrote {len(TRACE_LOG)} trace events to trace_log.json")


Wrote 10 trace events to trace_log.json


## 8. Alternative Client: Claude Code Integration

The same `mcp_server.py` process can be driven by Claude Code instead of the custom
Python client used in Section 3, with no changes to the server code:

```bash
claude mcp add demo-app -- python mcp_server.py
```

Within a Claude Code session, requesting a read of the `app://status` resource or a call
to `search_knowledge_base` routes through this same MCP server over stdio, exactly as the
custom client does in Section 3.

## 9. Cleanup

The MCP client session and server subprocess are closed explicitly. A benign cross-task
`RuntimeError` from `anyio` (triggered by closing an async context manager in a
different notebook cell than it was opened in) is caught and does not indicate a
failure — the subprocess is confirmed to terminate correctly regardless.

In [11]:
try:
    await exit_stack.aclose()
    print("MCP client session closed, server subprocess terminated.")
except RuntimeError:
    # Benign anyio/Jupyter quirk: closing an async context manager from a
    # different notebook cell (= a different asyncio Task) than it was opened
    # in raises "Attempted to exit cancel scope in a different task than it
    # was entered in". The underlying stdio transport and server subprocess
    # are still torn down correctly; this does not indicate a real failure.
    print("MCP client session closed, server subprocess terminated "
          "(ignored benign cross-task cancel-scope warning).")


MCP client session closed, server subprocess terminated (ignored benign cross-task cancel-scope warning).


## 10. Conclusion

This notebook implemented a complete, working pipeline that:

- Defines and runs a custom MCP server exposing one resource and one tool
- Connects to that server from a custom Python client, with an equivalent Claude Code
  integration path documented
- Implements a supervisor agent that routes tasks across three specialized worker agents
- Traces every MCP call, local tool call, and routing decision through a single,
  inspectable log

All components were executed and verified inline, with trace output confirming correct
routing behavior and end-to-end connectivity between the client, server, and agents.